# Phase 1: Foundation — Skeleton Graph

Build the AI Travel Genie multi-agent graph with stub agents returning mock data.
Demonstrates: StateGraph, conditional routing, supervisor pattern, graph visualization.

## 1. Setup — Add `src/` to Python Path

In [ ]:
import sys
import os

# Add project root to path so we can import from src/
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

## 2. Review the State Definition

In [ ]:
from src.state import TravelPlanState, TravelRequest

# Display the state fields
print("TravelRequest fields:")
for field, ftype in TravelRequest.__annotations__.items():
    print(f"  {field}: {ftype}")

print("\nTravelPlanState fields:")
for field, ftype in TravelPlanState.__annotations__.items():
    print(f"  {field}: {ftype}")

## 3. Build the LangGraph StateGraph

Wire all nodes: Intent Parser → Supervisor → [5 Agents] → Aggregator

In [ ]:
from langgraph.graph import StateGraph, START, END

from src.supervisor import parse_intent, supervisor_router, aggregate_results
from src.agents.destination_agent import destination_research_agent
from src.agents.flight_agent import flight_search_agent
from src.agents.hotel_agent import hotel_search_agent
from src.agents.activity_agent import activity_itinerary_agent
from src.agents.budget_agent import budget_currency_agent
from src.state import TravelPlanState

# Build the graph
builder = StateGraph(TravelPlanState)

# Add nodes
builder.add_node("intent_parser", parse_intent)
builder.add_node("destination_research", destination_research_agent)
builder.add_node("flight_search", flight_search_agent)
builder.add_node("hotel_search", hotel_search_agent)
builder.add_node("activity_itinerary", activity_itinerary_agent)
builder.add_node("budget_currency", budget_currency_agent)
builder.add_node("aggregator", aggregate_results)

# Wire edges
builder.add_edge(START, "intent_parser")
builder.add_conditional_edges("intent_parser", supervisor_router)
builder.add_conditional_edges("destination_research", supervisor_router)
builder.add_conditional_edges("flight_search", supervisor_router)
builder.add_conditional_edges("hotel_search", supervisor_router)
builder.add_conditional_edges("activity_itinerary", supervisor_router)
builder.add_conditional_edges("budget_currency", supervisor_router)
builder.add_edge("aggregator", END)

# Compile
graph = builder.compile()
print("Graph compiled successfully!")

## 4. Visualize the Graph

In [ ]:
from IPython.display import Image, display

try:
    img = graph.get_graph().draw_mermaid_png()
    display(Image(img))
except Exception as e:
    # Fallback: print the Mermaid diagram as text
    print("Could not render PNG (requires playwright or pyppeteer).")
    print("Mermaid diagram:\n")
    print(graph.get_graph().draw_mermaid())

## 5. Run the Graph — Bali Sample Request

Test with: *"Plan a 5-day beach vacation in Bali for 2 people, budget $3000, departing from New York"*

In [ ]:
from langchain_core.messages import HumanMessage

initial_state = {
    "messages": [
        HumanMessage(
            content="Plan a 5-day beach vacation in Bali for 2 people, "
            "budget $3000, departing from New York. "
            "Dates: March 15-20, 2025."
        )
    ],
}

print("Running AI Travel Genie...\n")
result = graph.invoke(initial_state)

# Print the final plan (last AI message)
for msg in result["messages"]:
    if hasattr(msg, "content"):
        print(msg.content)
        print()

## 6. Inspect Individual Agent Outputs

In [ ]:
import json

print("=== TRAVEL REQUEST ===")
print(json.dumps(result["travel_request"], indent=2))

print("\n=== DESTINATION RESEARCH ===")
print(json.dumps(result["destination_research"], indent=2))

print("\n=== FLIGHT OPTIONS ===")
print(json.dumps(result["flight_options"], indent=2))

print("\n=== HOTEL OPTIONS ===")
print(json.dumps(result["hotel_options"], indent=2))

print("\n=== ITINERARY ===")
print(json.dumps(result["itinerary"], indent=2))

print("\n=== BUDGET SUMMARY ===")
print(json.dumps(result["budget_summary"], indent=2))

print(f"\n=== PLAN STATUS: {result['plan_status']} ===")

## 7. Step-by-Step Execution (See Each Agent Working)

In [ ]:
from langchain_core.messages import HumanMessage

initial_state = {
    "messages": [
        HumanMessage(
            content="Plan a 7-day romantic getaway to the Maldives for 2 people. "
            "Budget is $5000. We love snorkeling and spa treatments. "
            "Flying from Chicago. Dates: March 1-8, 2025."
        )
    ],
}

print("Stepping through the graph...\n")
for step in graph.stream(initial_state):
    node_name = list(step.keys())[0]
    print(f"--- Node: {node_name} ---")
    node_output = step[node_name]
    if "messages" in node_output:
        for msg in node_output["messages"]:
            if hasattr(msg, "content"):
                print(f"  {msg.content}")
    if "plan_status" in node_output:
        print(f"  Status: {node_output['plan_status']}")
    print()

## Summary

Phase 1 complete. We have:
- **TravelPlanState** defined with all fields
- **StateGraph** with 7 nodes and conditional routing
- **5 stub agents** returning realistic mock data
- **Supervisor router** that dispatches based on missing plan sections
- **Graph visualization** showing the architecture
- **End-to-end test** with both Bali and Maldives requests

Next: Phase 2 — Replace the Destination stub with a real agent using Tavily web search.